# Agent-Based Model of Cancer Stem Cell (CSC) Driven Tumor Growth

This Jupyter notebook implements a 2D agent-based model (ABM) of tumor growth driven by cancer stem cells (CSC) and progenitor/differentiated cells (PC). It includes three requested extensions:

1. **Therapy pulse** (increased death rates during a time window).
2. **Nutrient diffusion PDE** that modulates division rates.
3. **Asynchronous Gillespie-style event simulation** (next-event sampling) and GIF creation of lattice snapshots.

The notebook is organized as follows:

* Introduction and model description (markdown)
* Parameters and imports (code)
* Helper functions (code)
* Gillespie ABM + nutrient PDE + therapy (code)
* GIF creation and visualization (code)
* Suggested experiments and extension ideas (markdown)

In [1]:
import numpy as np
import random, math, os, tempfile, time as pytime
import matplotlib.pyplot as plt
from matplotlib import colors
import imageio

In [3]:
# ==============================
# === Core Simulation Classes ==
# ==============================

class Parameters:
    """Holds all constants for the simulation."""
    def __init__(self):
        # Lattice
        self.Lx, self.Ly = 60, 60
        self.seed = 12
        np.random.seed(self.seed)
        random.seed(self.seed)

        # Cell types
        self.EMPTY, self.CSC, self.PC = 0, 1, 2

        # Rates
        self.rate_div_CSC = 0.12
        self.rate_div_PC = 0.20
        self.rate_death_CSC = 0.002
        self.rate_death_PC = 0.01
        self.rate_move = 0.04

        # CSC division outcomes
        self.p_sym_self = 0.12
        self.p_sym_diff = 0.04
        self.pc_max_divisions = 3

        # Nutrient PDE parameters
        self.use_nutrient = True
        self.D = 1.0
        self.decay = 0.01
        self.uptake = 0.02
        self.nutrient_source = 1.0
        self.nutrient_dt = 0.4

        # Therapy parameters
        self.use_therapy = True
        self.therapy_start = 10.0
        self.therapy_end = 18.0
        self.therapy_fold_increase_PC_death = 20.0
        self.therapy_fold_increase_CSC_death = 5.0

        # Simulation controls
        self.t_max = 2000.
        self.frames_count = 100
        self.snapshot_times = np.linspace(0, self.t_max, self.frames_count)
        self.out_gif = 'output/tumor_dynamics.gif'


class Lattice:
    """Represents the lattice grid, containing cells and nutrients."""
    def __init__(self, params: Parameters):
        self.p = params
        self.cell_type = np.zeros((self.p.Lx, self.p.Ly), dtype=np.int8)
        self.pc_div_left = np.zeros((self.p.Lx, self.p.Ly), dtype=np.int8)
        self.nutrient = np.ones((self.p.Lx, self.p.Ly)) * self.p.nutrient_source if self.p.use_nutrient else None
        self.nbrs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

        # Place initial cells
        cx, cy = self.p.Lx // 2, self.p.Ly // 3
        self.cell_type[cx-5, cy] = self.p.CSC
        self.cell_type[cx+5, cy] = self.p.CSC
        self.pc_div_left[:] = 0
        self.pc_div_left[self.cell_type == self.p.PC] = self.p.pc_max_divisions

    def in_bounds(self, x, y):
        return 0 <= x < self.p.Lx and 0 <= y < self.p.Ly

    def get_empty_neighbors(self, x, y):
        coords = []
        for dx, dy in self.nbrs:
            nx, ny = x + dx, y + dy
            if self.in_bounds(nx, ny) and self.cell_type[nx, ny] == self.p.EMPTY:
                coords.append((nx, ny))
        return coords

    def nutrient_step(self, dt):
        """Explicit finite-difference diffusion + decay + uptake."""
        if self.nutrient is None:
            return
        new = self.nutrient.copy()
        lap = np.zeros_like(self.nutrient)
        lap[1:-1, 1:-1] = (
            self.nutrient[2:, 1:-1] + self.nutrient[:-2, 1:-1] +
            self.nutrient[1:-1, 2:] + self.nutrient[1:-1, :-2] -
            4 * self.nutrient[1:-1, 1:-1]
        )
        uptake_field = np.zeros_like(self.nutrient)
        uptake_field[self.cell_type != self.p.EMPTY] = self.p.uptake
        new += dt * (self.p.D * lap - self.p.decay * self.nutrient - uptake_field)
        new[0, :] = self.p.nutrient_source
        new[-1, :] = self.p.nutrient_source
        new[:, 0] = self.p.nutrient_source
        new[:, -1] = self.p.nutrient_source
        new[new < 0] = 0.0
        self.nutrient = new


class EventSystem:
    """Handles the stochastic event selection using the Gillespie algorithm."""
    def __init__(self, lattice: Lattice):
        self.lattice = lattice
        self.p = lattice.p

    def compute_propensities(self, current_time):
        events, rates = [], []
        if self.p.use_therapy and self.p.therapy_start <= current_time <= self.p.therapy_end:
            rd_pc = self.p.rate_death_PC * self.p.therapy_fold_increase_PC_death
            rd_csc = self.p.rate_death_CSC * self.p.therapy_fold_increase_CSC_death
        else:
            rd_pc, rd_csc = self.p.rate_death_PC, self.p.rate_death_CSC

        xs, ys = np.nonzero(self.lattice.cell_type != self.p.EMPTY)
        for x, y in zip(xs, ys):
            c = self.lattice.cell_type[x, y]
            nut_factor = 1.0
            if self.p.use_nutrient:
                local_n = self.lattice.nutrient[x, y]
                nut_factor = local_n / (0.5 + local_n)

            if c == self.p.CSC:
                rdiv, rdeath, rmove = self.p.rate_div_CSC * nut_factor, rd_csc, self.p.rate_move
            elif c == self.p.PC:
                rdiv = self.p.rate_div_PC * nut_factor if self.lattice.pc_div_left[x, y] > 0 else 0.0
                rdeath, rmove = rd_pc, self.p.rate_move * 0.5
            else:
                continue

            if self.lattice.get_empty_neighbors(x, y):
                if rdiv > 0: events.append(('div', x, y)); rates.append(rdiv)
                events.append(('move', x, y)); rates.append(rmove)
            events.append(('death', x, y)); rates.append(rdeath)

        if not rates:
            return [], [], 0.0
        rates = np.array(rates, dtype=float)
        return events, rates, rates.sum()


class TumorSimulation:
    """Main simulation class controlling time evolution and output."""
    def __init__(self, params: Parameters):
        self.p = params
        self.lattice = Lattice(params)
        self.event_system = EventSystem(self.lattice)
        self.t, self.event_count = 0.0, 0
        self.max_events = 2_000_000
        self.tempdir = tempfile.mkdtemp()
        self.frames, self.frame_idx = [], 0
        self.next_snapshot_idx = 0
        self.data_records = []  # Store state data at each step

    def record_state(self):
        """Store system state (cell counts, mean nutrient, etc.) each time step."""
        csc_count = np.sum(self.lattice.cell_type == self.p.CSC)
        pc_count = np.sum(self.lattice.cell_type == self.p.PC)
        mean_nutrient = np.mean(self.lattice.nutrient) if self.p.use_nutrient else 0
        self.data_records.append((self.t, csc_count, pc_count, mean_nutrient))

    def capture_frame(self):
        cmap = colors.ListedColormap(['white', 'red', 'blue'])
        bounds = [0, 0.5, 1.5, 2.5]
        norm = colors.BoundaryNorm(bounds, cmap.N)
        fig = plt.figure(figsize=(4, 4), dpi=100)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(self.lattice.cell_type.T, origin='lower', cmap=cmap, norm=norm)
        ax.text(2, 2, f"t={self.t:.2f}", color='black', bbox=dict(facecolor='white', alpha=0.6), fontsize=8)
        fname = os.path.join(self.tempdir, f"frame_{self.frame_idx:04d}.png")
        fig.savefig(fname, dpi=100)
        plt.close(fig)
        self.frames.append(fname)
        self.frame_idx += 1

    def run(self):
        start_wall = pytime.time()
        self.capture_frame()
        self.record_state()

        while self.t < self.p.t_max and self.event_count < self.max_events:
            events, rates, total_rate = self.event_system.compute_propensities(self.t)
            if total_rate <= 0:
                print(f"Stopped: no more events at t={self.t}")
                break

            dt = -math.log(np.random.rand()) / total_rate
            self.t += dt

            # Nutrient update
            if self.p.use_nutrient:
                num_steps = max(1, int(math.ceil(dt / self.p.nutrient_dt)))
                small_dt = dt / num_steps
                for _ in range(num_steps):
                    self.lattice.nutrient_step(small_dt)

            # Choose and perform event
            ev = random.choices(events, weights=rates, k=1)[0]
            self.handle_event(ev)
            self.event_count += 1
            self.record_state()

            # Capture scheduled frames
            if self.next_snapshot_idx < len(self.p.snapshot_times) and self.t >= self.p.snapshot_times[self.next_snapshot_idx]:
                self.capture_frame()
                self.next_snapshot_idx += 1

        self.capture_frame()
        end_wall = pytime.time()
        print(f"Simulation finished: t={self.t:.2f}, events={self.event_count}, wall_time={end_wall - start_wall:.1f}s")

        self.save_gif()
        self.save_data()

    def handle_event(self, ev):
        etype, x, y = ev
        cell_type = self.lattice.cell_type
        pc_div_left = self.lattice.pc_div_left
        if etype == 'death':
            cell_type[x, y] = self.p.EMPTY
            pc_div_left[x, y] = 0
        elif etype == 'move':
            empty_nb = self.lattice.get_empty_neighbors(x, y)
            if empty_nb:
                nx, ny = random.choice(empty_nb)
                cell_type[nx, ny] = cell_type[x, y]
                pc_div_left[nx, ny] = pc_div_left[x, y]
                cell_type[x, y] = self.p.EMPTY
                pc_div_left[x, y] = 0
        elif etype == 'div':
            empty_nb = self.lattice.get_empty_neighbors(x, y)
            if not empty_nb: return
            nx, ny = random.choice(empty_nb)
            if cell_type[x, y] == self.p.CSC:
                r = np.random.rand()
                if r < self.p.p_sym_self:
                    cell_type[nx, ny] = self.p.CSC
                    pc_div_left[nx, ny] = 0
                elif r < self.p.p_sym_self + self.p.p_sym_diff:
                    cell_type[x, y] = self.p.PC
                    cell_type[nx, ny] = self.p.PC
                    pc_div_left[x, y] = pc_div_left[nx, ny] = self.p.pc_max_divisions - 1
                else:
                    cell_type[nx, ny] = self.p.PC
                    pc_div_left[nx, ny] = self.p.pc_max_divisions - 1
            elif cell_type[x, y] == self.p.PC and pc_div_left[x, y] > 0:
                cell_type[nx, ny] = self.p.PC
                pc_div_left[nx, ny] = pc_div_left[x, y] - 1
                pc_div_left[x, y] -= 1

    def save_gif(self):
        images = [imageio.imread(fname) for fname in self.frames]
        os.makedirs(os.path.dirname(self.p.out_gif), exist_ok=True)
        imageio.mimsave(self.p.out_gif, images, duration=3.0)
        print(f"Saved GIF to {self.p.out_gif}")

    def save_data(self):
        np.savetxt("output/simulation_data.csv", self.data_records, delimiter=",",
                   header="time,CSC_count,PC_count,mean_nutrient", comments="")
        print("Saved data to output/simulation_data.csv")


In [4]:
p = Parameters()
sim = TumorSimulation(p)
sim.run()

Simulation finished: t=2000.00, events=15767, wall_time=229.4s


/tmp/ipykernel_112873/3566163156.py:251: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(fname) for fname in self.frames]


Saved GIF to output/tumor_dynamics.gif
Saved data to output/simulation_data.csv
